In [2]:
!pip install rdflib

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [15]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL

In [ ]:
ontology = Graph()

In [ ]:
KG = Namespace("http://example.org/kg/")

In [ ]:
ontology.add((KG.Person, RDF.type, OWL.Class))
ontology.add((KG.Profession, RDF.type, OWL.Class))
ontology.add((KG.Film, RDF.type, OWL.Class))

<Graph identifier=N9fb5bda5e9e644abb5e962176ae55138 (<class 'rdflib.graph.Graph'>)>

In [ ]:
ontology.add((KG.hasProfession, RDF.type, OWL.ObjectProperty))

<Graph identifier=N9fb5bda5e9e644abb5e962176ae55138 (<class 'rdflib.graph.Graph'>)>

In [ ]:
ontology.add((KG.hasProfession, RDFS.domain, KG.Person))
ontology.add((KG.hasProfession, RDFS.range, KG.Profession))

<Graph identifier=N9fb5bda5e9e644abb5e962176ae55138 (<class 'rdflib.graph.Graph'>)>

In [ ]:
RDFS.domain, KG.Person

(rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#domain'),
 rdflib.term.URIRef('http://example.org/kg/Person'))

In [ ]:
RDFS.range, KG.Profession

(rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#range'),
 rdflib.term.URIRef('http://example.org/kg/Profession'))

In [ ]:
ontology.add((KG.actedIn, RDF.type, OWL.ObjectProperty))

ontology.add((KG.actedIn, RDFS.domain, KG.Person))
ontology.add((KG.actedIn, RDFS.range, KG.Film))

<Graph identifier=N9fb5bda5e9e644abb5e962176ae55138 (<class 'rdflib.graph.Graph'>)>

In [ ]:
for subject, predicate, object_ in ontology:
    print(subject, "→", predicate, "→", object_)

http://example.org/kg/actedIn → http://www.w3.org/2000/01/rdf-schema#domain → http://example.org/kg/Person
http://example.org/kg/hasProfession → http://www.w3.org/2000/01/rdf-schema#range → http://example.org/kg/Profession
http://example.org/kg/hasProfession → http://www.w3.org/2000/01/rdf-schema#domain → http://example.org/kg/Person
http://example.org/kg/actedIn → http://www.w3.org/1999/02/22-rdf-syntax-ns#type → http://www.w3.org/2002/07/owl#ObjectProperty
http://example.org/kg/Profession → http://www.w3.org/1999/02/22-rdf-syntax-ns#type → http://www.w3.org/2002/07/owl#Class
http://example.org/kg/actedIn → http://www.w3.org/2000/01/rdf-schema#range → http://example.org/kg/Film
http://example.org/kg/Person → http://www.w3.org/1999/02/22-rdf-syntax-ns#type → http://www.w3.org/2002/07/owl#Class
http://example.org/kg/hasProfession → http://www.w3.org/1999/02/22-rdf-syntax-ns#type → http://www.w3.org/2002/07/owl#ObjectProperty
http://example.org/kg/Film → http://www.w3.org/1999/02/22-rdf-

In [ ]:
import pandas as pd

train = pd.read_csv(
    "../dataset/train.txt",
    sep="\t",
    header=None,
    names=["head", "relation", "tail"]
)

print("Training triples:", len(train))

Training triples: 272115


In [ ]:
profession_triples = train[
    train["relation"] == "/people/person/profession"
]

profession_triples.head(10)

,head,relation,tail
21,/m/0grwj,/people/person/profession,/m/05sxg2
31,/m/053y0s,/people/person/profession,/m/0dz3r
70,/m/0411q,/people/person/profession,/m/016z4k
117,/m/02g8h,/people/person/profession,/m/01d_h8
126,/m/042l3v,/people/person/profession,/m/01d_h8
133,/m/0hl3d,/people/person/profession,/m/0cbd2
215,/m/06cv1,/people/person/profession,/m/02jknp
237,/m/01nqfh_,/people/person/profession,/m/01c8w0
288,/m/0kzy0,/people/person/profession,/m/0gbbt
368,/m/03_gd,/people/person/profession,/m/07s467s


In [ ]:
triple = profession_triples.iloc[0]

head = triple["head"]
tail = triple["tail"]

print("Head:", head)
print("Relation:", triple["relation"])
print("Tail:", tail)

Head: /m/0grwj
Relation: /people/person/profession
Tail: /m/05sxg2


In [ ]:
truths = [
    f"Person({head})",
    f"Profession({tail})",
    f"hasProfession({head}, {tail})"
]

for truth in truths:
    print(truth)

Person(/m/0grwj)
Profession(/m/05sxg2)
hasProfession(/m/0grwj, /m/05sxg2)


In [ ]:
rule = {
    "name": "Profession Type Rule",
    "condition": "hasProfession(X, Y)",
    "conclusion": "Profession(Y)"
}

print(rule)

{'name': 'Profession Type Rule', 'condition': 'hasProfession(X, Y)', 'conclusion': 'Profession(Y)'}


In [ ]:
# Our existing fact
fact = {
    "relation": "hasProfession",
    "head": "/m/0grwj",
    "tail": "/m/05sxg2"
}

# Apply the rule
if fact["relation"] == "hasProfession":
    inferred_truth = f"Profession({fact['tail']})"

print("Original fact:")
print(f"{fact['relation']}({fact['head']}, {fact['tail']})")

print("\nInferred truth:")
print(inferred_truth)

Original fact:
hasProfession(/m/0grwj, /m/05sxg2)

Inferred truth:
Profession(/m/05sxg2)


In [ ]:
# Show some relations in the dataset
relations = train["relation"].unique()

for relation in relations[:30]:
    print(relation)

/location/country/form_of_government
/tv/tv_program/regular_cast./tv/regular_tv_appearance/actor
/media_common/netflix_genre/titles
/award/award_winner/awards_won./award/award_honor/award_winner
/soccer/football_team/current_roster./sports/sports_team_roster/position
/soccer/football_team/current_roster./soccer/football_roster_position/position
/film/actor/film./film/performance/film
/award/award_category/nominees./award/award_nomination/nominated_for
/award/award_nominee/award_nominations./award/award_nomination/award_nominee
/music/performance_role/regular_performances./music/group_membership/role
/award/award_category/winners./award/award_honor/ceremony
/film/film/release_date_s./film/film_regional_release_date/film_release_distribution_medium
/award/award_winning_work/awards_won./award/award_honor/award_winner
/film/film/release_date_s./film/film_regional_release_date/film_release_region
/film/film/language
/location/location/contains
/organization/organization/headquarters./locati

In [ ]:
# Look at examples of the actor-film relation

actor_film = train[
    train["relation"] == "/film/actor/film./film/performance/film"
]

actor_film.head(10)

,head,relation,tail
6,/m/07nznf,/film/actor/film./film/performance/film,/m/014lc_
17,/m/079vf,/film/actor/film./film/performance/film,/m/0d90m
38,/m/012d40,/film/actor/film./film/performance/film,/m/034qmv
54,/m/04yywz,/film/actor/film./film/performance/film,/m/0140g4
57,/m/06688p,/film/actor/film./film/performance/film,/m/0b2v79
91,/m/06dv3,/film/actor/film./film/performance/film,/m/09m6kg
111,/m/049tjg,/film/actor/film./film/performance/film,/m/05jf85
144,/m/023tp8,/film/actor/film./film/performance/film,/m/0dq626
158,/m/09fb5,/film/actor/film./film/performance/film,/m/07xtqq
173,/m/032xhg,/film/actor/film./film/performance/film,/m/034qrh


In [ ]:
# Choose the film/entity from our first example
entity = "/m/014lc_"

# Find all triples where this entity appears as the HEAD
connected = train[train["head"] == entity]

connected.head(20)

,head,relation,tail
743,/m/014lc_,/film/film/release_date_s./film/film_regional_...,/m/06mzp
2305,/m/014lc_,/film/film/language,/m/02h40lc
9622,/m/014lc_,/film/film/release_date_s./film/film_regional_...,/m/0hzlz
9763,/m/014lc_,/film/film/genre,/m/03k9fj
16493,/m/014lc_,/film/film/release_date_s./film/film_regional_...,/m/05qhw
18789,/m/014lc_,/film/film/release_date_s./film/film_regional_...,/m/06qd3
22588,/m/014lc_,/film/film/release_date_s./film/film_regional_...,/m/0d060g
23159,/m/014lc_,/film/film/release_date_s./film/film_regional_...,/m/09c7w0
29522,/m/014lc_,/film/film/release_date_s./film/film_regional_...,/m/02vzc
45270,/m/014lc_,/film/film/release_date_s./film/film_regional_...,/m/0f8l9c


In [ ]:
film_genre = train[
    train["relation"] == "/film/film/genre"
]

film_genre.head(10)

,head,relation,tail
44,/m/0sxg4,/film/film/genre,/m/04xvlr
73,/m/06w99h3,/film/film/genre,/m/02kdv5l
79,/m/027qgy,/film/film/genre,/m/06cvj
89,/m/047q2k1,/film/film/genre,/m/05p553
95,/m/06wzvr,/film/film/genre,/m/01q03
141,/m/0ddfwj1,/film/film/genre,/m/07s9rl0
159,/m/01k1k4,/film/film/genre,/m/05p553
241,/m/060v34,/film/film/genre,/m/03npn
258,/m/04ddm4,/film/film/genre,/m/02n4kr
260,/m/03s6l2,/film/film/genre,/m/0lsxr


In [ ]:
# Two facts from our KG

fact1 = {
    "relation": "actorFilm",
    "head": "/m/07nznf",
    "tail": "/m/014lc_"
}

fact2 = {
    "relation": "filmGenre",
    "head": "/m/014lc_",
    "tail": "/m/0lsxr"
}

# Check whether the two facts can be connected
if fact1["tail"] == fact2["head"]:

    inferred_edge = (
        fact1["head"],
        "actorGenre",
        fact2["tail"]
    )

    print("Fact 1:")
    print(f"{fact1['head']} --{fact1['relation']}--> {fact1['tail']}")

    print("\nFact 2:")
    print(f"{fact2['head']} --{fact2['relation']}--> {fact2['tail']}")

    print("\nInferred new edge:")
    print(
        f"{inferred_edge[0]} --{inferred_edge[1]}--> {inferred_edge[2]}"
    )

Fact 1:
/m/07nznf --actorFilm--> /m/014lc_

Fact 2:
/m/014lc_ --filmGenre--> /m/0lsxr

Inferred new edge:
/m/07nznf --actorGenre--> /m/0lsxr


In [ ]:
# -----------------------------
# Candidate B: Language
# -----------------------------

# Fact 1: actor -> film
fact1 = {
    "head": "/m/07nznf",
    "relation": "/film/actor/film./film/performance/film",
    "tail": "/m/014lc_"
}

# Find all facts where the film is the HEAD
film = fact1["tail"]

connected = train[train["head"] == film]

# Look specifically for the film -> language relationship
language_facts = connected[
    connected["relation"] == "/film/film/language"
]

print("Fact 1:")
print(f"{fact1['head']} --actorFilm--> {fact1['tail']}")

print("\nLanguage facts connected to this film:")

for _, row in language_facts.iterrows():
    print(
        f"{row['head']} --filmLanguage--> {row['tail']}"
    )

Fact 1:
/m/07nznf --actorFilm--> /m/014lc_

Language facts connected to this film:
/m/014lc_ --filmLanguage--> /m/02h40lc


In [ ]:
# Create an inferred actor-language edge

for _, row in language_facts.iterrows():

    inferred_edge = {
        "head": fact1["head"],
        "relation": "actorLanguage",
        "tail": row["tail"]
    }

    print("\nInferred candidate edge:")
    print(
        f"{inferred_edge['head']} "
        f"--{inferred_edge['relation']}--> "
        f"{inferred_edge['tail']}"
    )


Inferred candidate edge:
/m/07nznf --actorLanguage--> /m/02h40lc


In [ ]:
# The intermediate entity from Fact 1
entity = "/m/014lc_"

# Find all triples where this entity is the HEAD
connected = train[train["head"] == entity]

# Show relation and tail
for _, row in connected.iterrows():
    print(
        "Intermediate:",
        row["head"],
        "| Relation:",
        row["relation"],
        "| Next entity:",
        row["tail"]
    )

Intermediate: /m/014lc_ | Relation: /film/film/release_date_s./film/film_regional_release_date/film_release_region | Next entity: /m/06mzp
Intermediate: /m/014lc_ | Relation: /film/film/language | Next entity: /m/02h40lc
Intermediate: /m/014lc_ | Relation: /film/film/release_date_s./film/film_regional_release_date/film_release_region | Next entity: /m/0hzlz
Intermediate: /m/014lc_ | Relation: /film/film/genre | Next entity: /m/03k9fj
Intermediate: /m/014lc_ | Relation: /film/film/release_date_s./film/film_regional_release_date/film_release_region | Next entity: /m/05qhw
Intermediate: /m/014lc_ | Relation: /film/film/release_date_s./film/film_regional_release_date/film_release_region | Next entity: /m/06qd3
Intermediate: /m/014lc_ | Relation: /film/film/release_date_s./film/film_regional_release_date/film_release_region | Next entity: /m/0d060g
Intermediate: /m/014lc_ | Relation: /film/film/release_date_s./film/film_regional_release_date/film_release_region | Next entity: /m/09c7w0
Inte

In [ ]:
%pip install networkx pyvis

   ---------------------------------------- 0.0/756.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/756.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/756.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/756.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/756.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/756.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/756.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/756.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/756.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/756.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/756.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/756.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/756.0 kB ? eta -:--:--
   ------------- -------------------------- 262.1/756.0 kB ? et

In [ ]:
import networkx as nx
from pyvis.network import Network
from IPython.display import display, HTML

In [ ]:

import networkx as nx

# Build the complete graph from ALL training triples
KG = nx.MultiDiGraph()

for head, relation, tail in train[
    ["head", "relation", "tail"]
].itertuples(index=False, name=None):
    KG.add_edge(
        head,
        tail,
        relation=relation
    )

print("Knowledge Graph created successfully")
print("Total unique entities:", KG.number_of_nodes())
print("Total relationship edges:", KG.number_of_edges())
print("Total training triples:", len(train))
print("Unique relations:", train["relation"].nunique())


Knowledge Graph created successfully
Total unique entities: 14505
Total relationship edges: 272115
Total training triples: 272115
Unique relations: 237


In [ ]:

# Save the complete graph in GraphML format
nx.write_graphml(KG, "FB15K237_complete_KG.graphml")

print("Saved: FB15K237_complete_KG.graphml")


Saved: FB15K237_complete_KG.graphml


In [ ]:

import networkx as nx
import os

KG = nx.MultiDiGraph()

for head, relation, tail in train[
    ["head", "relation", "tail"]
].itertuples(index=False, name=None):
    KG.add_edge(head, tail, relation=relation)

output_file = os.path.abspath("FB15K237_complete_KG.graphml")
nx.write_graphml(KG, output_file)

print("Graph saved to:", output_file)
print("Entities:", KG.number_of_nodes())
print("Edges:", KG.number_of_edges())


Graph saved to: C:\Users\sugit\OneDrive\Tài liệu\KG_Ontology_Project\notebooks\FB15K237_complete_KG.graphml
Entities: 14505
Edges: 272115


In [ ]:

import networkx as nx
import os

# Build the complete graph from all training triples
KG = nx.MultiDiGraph()

for i, (head, relation, tail) in enumerate(
    train[["head", "relation", "tail"]]
    .itertuples(index=False, name=None)
):
    KG.add_edge(
        head,
        tail,
        key=str(i),
        id=str(i),
        relation=relation
    )

# Save using the unique edge ID attribute
output_file = os.path.abspath(
    "FB15K237_complete_KG_fixed.graphml"
)

nx.write_graphml(
    KG,
    output_file,
    edge_id_from_attribute="id"
)

print("Saved to:", output_file)
print("Nodes:", KG.number_of_nodes())
print("Edges:", KG.number_of_edges())
print("Training triples:", len(train))

# Verify the exported file
check = nx.read_graphml(
    output_file,
    force_multigraph=True
)

print("Export verification — nodes:", check.number_of_nodes())
print("Export verification — edges:", check.number_of_edges())


Saved to: C:\Users\sugit\OneDrive\Tài liệu\KG_Ontology_Project\notebooks\FB15K237_complete_KG_fixed.graphml
Nodes: 14505
Edges: 272115
Training triples: 272115
Export verification — nodes: 14505
Export verification — edges: 272115


In [ ]:
check = nx.read_graphml(
    "FB15K237_complete_KG_fixed.graphml",
    force_multigraph=True
)

print("Exported nodes:", check.number_of_nodes())
print("Exported edges:", check.number_of_edges())

Exported nodes: 14505
Exported edges: 272115


In [6]:
import sys
print(sys.executable)

c:\ProgramData\anaconda3\python.exe


In [7]:

import sys
!{sys.executable} -m pip install rdflib


Defaulting to user installation because normal site-packages is not writeable


In [9]:
import sys
!"{sys.executable}" -m pip install --user --upgrade rdflib

In [10]:
import sys
import site

print("Python:", sys.executable)
print("User packages:", site.getusersitepackages())
print("User site enabled:", site.ENABLE_USER_SITE)

Python: c:\ProgramData\anaconda3\python.exe
User packages: C:\Users\sugit\AppData\Roaming\Python\Python313\site-packages
User site enabled: True


In [12]:

import sys

# Add the folder where RDFLib was installed
user_packages = r"C:\Users\sugit\AppData\Roaming\Python\Python313\site-packages"

if user_packages not in sys.path:
    sys.path.insert(0, user_packages)

# Test the import
import rdflib

print("RDFLib version:", rdflib.__version__)
print("RDFLib location:", rdflib.__file__)


RDFLib version: 7.6.0
RDFLib location: C:\Users\sugit\AppData\Roaming\Python\Python313\site-packages\rdflib\__init__.py


In [13]:

import pandas as pd
import networkx as nx
import rdflib

print("Pandas:", pd.__version__)
print("NetworkX:", nx.__version__)
print("RDFLib:", rdflib.__version__)

print("\nAll three libraries imported successfully!")


Pandas: 2.3.3
NetworkX: 3.5
RDFLib: 7.6.0

All three libraries imported successfully!


In [16]:

from pathlib import Path

project_dir = Path.cwd()
print("Current directory:", project_dir)

for p in project_dir.rglob("train.txt"):
    print("Found training file:", p)


Current directory: c:\Users\sugit\OneDrive\Tài liệu\KG_Ontology_Project\notebooks
